## Import libraries

In [1]:
import tomopy
from helperFunctions import MoviePlotter
from tomoDataClass import tomoData
from alignment_methods import reprojection_consistency_score
import h5py
import numpy as np
from skimage.transform import pyramid_gaussian

import os
os.environ['TOMOPY_PYTHON_THREADS'] = '80'
import tomopy

PyTorch imported successfully.
SVMBIR imported successfully.


In [2]:
def tomo_data(file,redo_align=False):
    try:
        with h5py.File(file) as hf:
            projs = hf['data'][...]
            angles = hf['angles'][...]
    except KeyError:
        with h5py.File(file) as hf:
            projs = hf['data'][...]
            angles = hf['angles'][...]
    angles = angles * np.pi / 180
    return projs, angles

## Import real Data

In [3]:
# Set projection angles
# filename = "/Users/levihancock/Library/CloudStorage/Box-Box/BYU_CXI_Research_Team/ProjectFolders/IFE-STAR/IFE-Ptycho-Tomo/APS_2ID_GUP1013052_August_2025/levi_tomoReconstructions/tomo_data_run_final_2.hdf5"
filename = "/home/ljh79/groups/grp_ptychi/nobackup/autodelete/Oct2025APSdata/tomo_data_run_final_2.hdf5"

projections_og, angles = tomo_data(filename, redo_align=True)
print(projections_og.shape)


(558, 585, 1810)


In [4]:

from scipy.ndimage import zoom

# #downsample projections
downsample = 4
print(projections_og.shape)
projections = zoom(projections_og, (1, 1/downsample, 1/downsample), order=1)
print(projections.shape)
# projections = projections_og[::2,::4,::4]
# angles = angles[::2]
# print("Angles:", angles)
# print(projections.shape)

# print("Projections through object")
# MoviePlotter(projections)

projections = projections_og
tomo = tomoData(projections, angles)

(558, 585, 1810)
(558, 146, 452)


In [5]:
print(projections.shape)
tomo.makeNotebookProjMovie()

(558, 585, 1810)


Output()

## Align Data

In [ ]:
#Show bad reconstruction prior to alignment
tomo.reset_workingProjections(x_size=None, y_size=None) #You can adjust these for tighter cropping
tomo.normalize(isPhaseData=True)

tomo.reconstruct(algorithm='fbp')
print("\nBad reconstruction prior to alignment")
badRecon = tomo.recon.copy()
MoviePlotter(badRecon)



Normalizing projections


Using CPU-based reconstruction. Algorithm:  fbp


In [ ]:
# # Reprojection Consistency Score — before alignment
# rcs_before, _, _ = reprojection_consistency_score(tomo, plot=True)

In [ ]:
tomo.reset_workingProjections(x_size=projections.shape[2]-(500//downsample), y_size=projections.shape[1], )
tomo.normalize(isPhaseData=True)

#Best XC params from sweep:
# Coarse passes: stepRatio=0.9 (stable global convergence)
tomo.cross_correlate_align(tolerance=0, maxShiftTolerance=0, max_iterations=10, stepRatio=0.9, yROI_Range=None, xROI_Range=None, isFull360=False, downsample=4, use_grad=True)
tomo.cross_correlate_align(tolerance=0, maxShiftTolerance=0, max_iterations=10, stepRatio=0.9, yROI_Range=None, xROI_Range=None, isFull360=False, downsample=2, use_grad=True)
# Final pass: stepRatio=0.75-0.8 with more iterations (fine-scale sharpness)
tomo.cross_correlate_align(tolerance=0, maxShiftTolerance=0, max_iterations=5, stepRatio=0.8, yROI_Range=None, xROI_Range=None, isFull360=False, downsample=1, use_grad=True)


#Best VMF params from sweep:
# tomo.vertical_mass_fluctuation_align(tolerance=0, max_iterations=5, y_range=None, sigma=None, smooth_sigma=1.0, window='soft_roi', roi_sigma=0.3, use_gradient=True, plot=True)

# tomo.PMA(max_iterations=5, tolerance=0, algorithm='SIRT_CUDA', levels=3, scale=4, iterations_per_level=[5,5,2], shift_method='optical_flow', of_sigma=3.0, plot=True)
tomo.make_updates_shift()



Normalizing projections


Cross-Correlation Alignment  [16x downsample | full frame | gradient mode]


Iteration 1/20:  19%|█▉        | 107/557 [00:13<00:58,  7.69it/s]


KeyboardInterrupt: 

In [ ]:
tomo.makeNotebookProjMovie()
print(tomo.ang)
tomo.reconstruct(algorithm='fbp')
print("\nGood Reconstruction after alignment")
tomo.makeNotebookReconMovie()

In [ ]:
# Reprojection Consistency Score — after alignment
rcs_after, _, _ = reprojection_consistency_score(tomo, plot=True)

# print("=" * 52)
# print("  ALIGNMENT QUALITY SUMMARY")
# print("=" * 52)
# print(f"  {'Metric':<22} {'Before':>8} {'After':>8} {'Δ':>10}")
# print(f"  {'-'*50}")
# print(f"  {'RCS (lower = better)':<22} {rcs_before:>8.4f} {rcs_after:>8.4f} {rcs_before - rcs_after:>+10.4f}")
# print("=" * 52)

In [ ]:
# import tifffile
# tifffile.imwrite("reconstruction(downsampled).tiff", tomo.recon.astype("float32"))